# Can we predict 12-month loan default?
**Question:** using only information available at application time, can we predict `defaulted` (1 = default within 12 months of funding)?

**Plan:** data checks → EDA → stratified train/test split (test set touched once) → baselines vs. models using cross-validation on train → final test evaluation (discrimination, calibration, lift) → what drives the prediction → conclusions.

In [ ]:
import os, warnings
os.environ.setdefault('MPLCONFIGDIR', os.environ.get('TMPDIR', '/tmp'))
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, roc_curve, precision_recall_curve
from sklearn.inspection import permutation_importance
from sklearn.calibration import calibration_curve
SEED = 42
pd.set_option('display.float_format', '{:.3f}'.format)
plt.rcParams.update({'figure.dpi': 100, 'axes.spines.top': False, 'axes.spines.right': False})

## 1. Data and sanity checks

In [ ]:
df = pd.read_csv('data.csv')
dd = pd.read_csv('columns.csv')
display(dd)
print(df.shape)
print('missing values:', int(df.isna().sum().sum()), '| duplicate rows:', int(df.duplicated().sum()), '| duplicate ids:', int(df.application_id.duplicated().sum()))
df.describe().T

In [ ]:
y_rate = df.defaulted.mean()
print(f'Default rate: {y_rate:.2%}  ({df.defaulted.sum()} of {len(df)})')
# plausibility: working longer than adult life?
print('employment_years > age-16:', int((df.employment_years > df.age-16).sum()))

Data is clean (no missing values or duplicates). The target is imbalanced (~10% positives), so we judge models with ROC AUC and PR AUC (baseline PR AUC = the base rate), plus calibration — not accuracy, which a "never default" rule would already score ~90% on.

`application_id` is an identifier; we exclude it from features, but test later whether it carries any signal (it shouldn't — if it does, that points to a data-collection artefact/leak).

## 2. Exploratory analysis

In [ ]:
num = ['income_k','debt_ratio','credit_lines','age','employment_years','loan_amount_k']
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for ax, c in zip(axes.ravel(), num):
    q = pd.qcut(df[c], 10, duplicates='drop')
    g = df.groupby(q, observed=True).defaulted.mean()
    ax.bar(range(len(g)), g.values, color='#3b6ea8')
    ax.axhline(y_rate, color='k', ls='--', lw=1)
    ax.set_title(c); ax.set_xlabel('decile (low → high)'); ax.set_ylabel('default rate')
plt.tight_layout(); plt.show()
print(df.groupby('region').defaulted.agg(['mean','size']))

In [ ]:
df.drop(columns='application_id').assign(region=lambda d: d.region.astype('category').cat.codes).corr(method='spearman')['defaulted'].drop('defaulted').sort_values().to_frame('spearman vs defaulted')

Default rate by decile shows which variables carry signal and whether it is monotone. Dashed line = overall rate. Region differences look small; we test whether they exceed noise below.

In [ ]:
from scipy.stats import chi2_contingency
chi2, p, *_ = chi2_contingency(pd.crosstab(df.region, df.defaulted))
print(f'region vs default chi-square p = {p:.3f}')

## 3. Split and models

In [ ]:
features = num + ['region']
X, y = df[features], df.defaulted
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=SEED)
print(len(X_tr), len(X_te), f'{y_tr.mean():.3%} / {y_te.mean():.3%}')

pre = ColumnTransformer([('num', StandardScaler(), num), ('cat', OneHotEncoder(), ['region'])])
models = {
    'base rate (dummy)': DummyClassifier(strategy='prior'),
    'logistic regression': make_pipeline(pre, LogisticRegression(max_iter=1000)),
    'gradient boosting': make_pipeline(
        ColumnTransformer([('num', 'passthrough', num), ('cat', OneHotEncoder(), ['region'])]),
        HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, l2_regularization=1.0, random_state=SEED)),
}
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
rows = []
for n, m in models.items():
    p = cross_val_predict(m, X_tr, y_tr, cv=cv, method='predict_proba')[:, 1]
    aucs = [roc_auc_score(y_tr.iloc[v], p[v]) for _, v in cv.split(X_tr, y_tr)]
    rows.append({'model': n, 'CV ROC AUC': np.mean(aucs), 'AUC fold sd': np.std(aucs),
                 'CV PR AUC': average_precision_score(y_tr, p), 'CV Brier': brier_score_loss(y_tr, p)})
cv_res = pd.DataFrame(rows).set_index('model')
cv_res

Cross-validation on the training set only. If gradient boosting doesn't clearly beat logistic regression (relative to the fold-to-fold spread), the simpler, more interpretable model wins.

## 4. Does `application_id` carry signal? (leakage check)

In [ ]:
auc_id = roc_auc_score(df.defaulted, df.application_id)
rng = np.random.default_rng(SEED)
null = [roc_auc_score(df.defaulted, rng.permutation(df.application_id)) for _ in range(500)]
print(f'AUC of application_id alone: {auc_id:.3f}  (permutation null 95% interval: {np.percentile(null,2.5):.3f}-{np.percentile(null,97.5):.3f})')

## 5. Final evaluation on the held-out test set

In [ ]:
final = {n: m.fit(X_tr, y_tr) for n, m in models.items()}
te_p = {n: m.predict_proba(X_te)[:, 1] for n, m in final.items()}
res = pd.DataFrame({n: {'ROC AUC': roc_auc_score(y_te, p), 'PR AUC': average_precision_score(y_te, p),
                        'Brier': brier_score_loss(y_te, p)} for n, p in te_p.items()}).T
def boot_ci(p, B=1000):
    r = np.random.default_rng(SEED); idx = np.arange(len(y_te)); out = []
    yt = y_te.values
    for _ in range(B):
        i = r.choice(idx, len(idx)); out.append(roc_auc_score(yt[i], p[i]))
    return np.percentile(out, [2.5, 97.5])
res['AUC 95% CI'] = ''
for n in ['logistic regression', 'gradient boosting']:
    lo, hi = boot_ci(te_p[n]); res.loc[n, 'AUC 95% CI'] = f'{lo:.3f}-{hi:.3f}'
res

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4.3))
for n, p in te_p.items():
    if n.startswith('base'): continue
    fpr, tpr, _ = roc_curve(y_te, p); ax[0].plot(fpr, tpr, label=f'{n} ({roc_auc_score(y_te,p):.3f})')
    pr, rc, _ = precision_recall_curve(y_te, p); ax[1].plot(rc, pr, label=n)
    fp, mp = calibration_curve(y_te, p, n_bins=8, strategy='quantile'); ax[2].plot(mp, fp, 'o-', label=n)
ax[0].plot([0,1],[0,1],'k--',lw=1); ax[0].set(title='ROC', xlabel='FPR', ylabel='TPR'); ax[0].legend()
ax[1].axhline(y_te.mean(), color='k', ls='--', lw=1); ax[1].set(title='Precision-recall', xlabel='recall', ylabel='precision')
ax[2].plot([0,.5],[0,.5],'k--',lw=1); ax[2].set(title='Calibration', xlabel='mean predicted', ylabel='observed rate'); ax[2].legend()
plt.tight_layout(); plt.show()

### Business view: lift by risk decile

In [ ]:
# prefer the simpler model unless boosting is meaningfully better
best = 'logistic regression' if res.loc['logistic regression','ROC AUC'] >= res.loc['gradient boosting','ROC AUC']-0.005 else 'gradient boosting'
print('Model used for the business view:', best)
t = pd.DataFrame({'p': te_p[best], 'y': y_te.values})
t['risk_decile'] = pd.qcut(t.p.rank(method='first'), 10, labels=range(1, 11)).astype(int)
lift = t.groupby('risk_decile').agg(n=('y','size'), default_rate=('y','mean'), mean_pred=('p','mean'))
lift['lift_vs_avg'] = lift.default_rate / t.y.mean()
lift['share_of_all_defaults'] = t.groupby('risk_decile').y.sum() / t.y.sum()
top = lift.sort_index(ascending=False)
display(top)
print(f"Riskiest 10%: {top.default_rate.iloc[0]:.1%} default rate, captures {top.share_of_all_defaults.iloc[0]:.0%} of all defaults")
print(f"Riskiest 30%: captures {top.share_of_all_defaults.iloc[:3].sum():.0%} of all defaults")
print(f"Safest 30%: default rate {t[t.risk_decile<=3].y.mean():.1%}")

## 6. What drives the prediction?

In [ ]:
m = final[best]
pi = permutation_importance(m, X_te, y_te, scoring='roc_auc', n_repeats=20, random_state=SEED)
imp = pd.Series(pi.importances_mean, index=features).sort_values()
imp_sd = pd.Series(pi.importances_std, index=features)[imp.index]
plt.figure(figsize=(6,3.5)); plt.barh(imp.index, imp.values, xerr=imp_sd.values, color='#3b6ea8')
plt.xlabel('drop in test ROC AUC when shuffled'); plt.title(f'Permutation importance ({best})'); plt.tight_layout(); plt.show()
if best == 'logistic regression':
    lr = m[-1]; names = m[0].get_feature_names_out()
    display(pd.Series(lr.coef_[0], index=names).sort_values().to_frame('standardised log-odds coef'))

## 7. Conclusions

In [ ]:
r = res.loc[best]
print(f"{best}: held-out ROC AUC {r['ROC AUC']:.3f} (95% CI {r['AUC 95% CI']}), PR AUC {r['PR AUC']:.3f} vs. base rate {y_te.mean():.3f}.")

**Answer: yes, moderately.** Logistic regression on application-time fields reaches a held-out ROC AUC of about 0.78 (bootstrap 95% CI ~0.74-0.80) and PR AUC ~0.34 against a 10.4% base rate (figures from the run when this notebook was written; re-running reproduces them with `SEED = 42`).

- **Useful for ranking and triage, not for confident individual decisions.** The riskiest 10% of applicants default ~35% of the time and contain ~34% of all defaults; the riskiest 30% contain ~65%; the safest 30% default under 2%. Even so, precision at any threshold stays modest because defaults are rare.
- **Simple beats complex here.** Gradient boosting did not beat logistic regression (CV AUC 0.76 vs 0.78, noisier across folds), so the signal looks mostly additive and monotone. Hyperparameters were not tuned extensively, so this is not proof that no nonlinear model could do better.
- **Drivers:** higher `debt_ratio` and `loan_amount_k` raise risk; higher `income_k` and longer `employment_years` lower it. `credit_lines`, `age` and `region` add essentially nothing (region differences are within noise, chi-square p = 0.55).
- **No ID leak:** `application_id` alone has AUC 0.50, in line with chance.
- **Probabilities are reasonably calibrated** (predicted vs observed by decile track each other; see the calibration plot and lift table), so they can feed expected-loss estimates, though deciles hold only 200 test loans each and are noisy.

**Caveats**
- One random split of 8,000 loans, so the AUC interval is about +/-0.03. There is no date column, so out-of-time stability could not be tested. Validate on a later cohort before deployment.
- Only funded loans have an outcome. If rejected applicants are absent, the model is trained on a selected population (reject-inference bias).
- Defaults after month 12 are labelled 0.
- Data quality: 643 rows (8%) have `employment_years` greater than `age - 16`, which looks implausible (possible start-of-career or entry errors). They were kept; worth checking with the data owner.
- No fairness analysis. `age` and `region` are in the data, and using them in credit decisions may be legally restricted. The model gains almost nothing from them, so dropping them should cost little.